In [30]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Row
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .appName("hive -> hbase (agg)")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

26/01/17 21:07:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/17 21:07:12 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [41]:
# Make sure the checkpoint table exists
spark.sql("""
CREATE TABLE IF NOT EXISTS cryptopredictions.batch_checkpoint (
    table_name STRING,
    last_processed_date DATE,
    run_ts TIMESTAMP
)
STORED AS PARQUET;
""")

26/01/17 21:28:43 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

In [61]:
# Read checkpoint for your table
checkpoint_df = (
    spark.table("cryptopredictions.batch_checkpoint")
    .filter(F.col("table_name") == "cryptocurrencysnapshot")
)

# Get the latest checkpoint
latest_checkpoint_row = checkpoint_df.orderBy(F.col("last_processed_date").desc()).limit(1).collect()

if latest_checkpoint_row:
    checkpoint = latest_checkpoint_row[0]["last_processed_date"]
else:
    checkpoint = None  # No checkpoint yet
print("Checkpoint:", checkpoint)
checkpoint = '2024-01-01'

crypto_df = spark.table("cryptopredictions.cryptocurrencysnapshot")
if checkpoint:
    crypto_df = crypto_df.filter(F.col("PartitionDate") > checkpoint)
        
    if crypto_df.count() == 0:
        print("No new data")

Checkpoint: 2026-01-16


In [62]:
# Sanity Check
bounds = (
    crypto_df.select(
        F.min("Datetime").alias("first_datetime"),
        F.max("Datetime").alias("last_datetime")
    )
    .collect()[0]
)

print(f"[SANITY CHECK] Datetime range: {bounds.first_datetime} →  {bounds.last_datetime}")

[Stage 72:======================================================> (32 + 1) / 33]

[SANITY CHECK] Datetime range: 2025-11-06 21:37:53 → 2026-01-16 23:19:07


In [63]:
def aggregate_crypto(df, window_duration, granularity_label):
    return (
        df
        .groupBy(
            "Symbol",
            F.window("Datetime", window_duration).alias("w")
        )
        .agg(
            F.first("CurrentPrice").alias("open"),
            F.max("CurrentPrice").alias("high"),
            F.min("CurrentPrice").alias("low"),
            F.last("CurrentPrice").alias("close")
        )
        .withColumn("granularity", F.lit(granularity_label))
        .withColumn("timestamp", F.col("w.start"))
        .drop("w")
    )
    
crypto_1m  = aggregate_crypto(crypto_df, "1 minute", "1m")
crypto_10m = aggregate_crypto(crypto_df, "10 minutes", "10m")
crypto_1d  = aggregate_crypto(crypto_df, "1 day", "1d")

def add_smas(df, periods=[7,30]):
    for period in periods:
        window_spec = Window.partitionBy("Symbol", "granularity").orderBy("timestamp").rowsBetween(-(period-1), 0)
        df = df.withColumn(f"SMA_{period}", F.avg("close").over(window_spec))
    return df

crypto_1m  = add_smas(crypto_1m)
crypto_10m = add_smas(crypto_10m)
crypto_1d  = add_smas(crypto_1d)

crypto_agg = crypto_1m.unionByName(crypto_10m).unionByName(crypto_1d)

crypto_agg.filter(F.col("granularity") == "1m").show(5)
crypto_agg.filter(F.col("granularity") == "10m").show(5)
crypto_agg.filter(F.col("granularity") == "1d").show(10)

+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|Symbol|     open|     high|      low|    close|granularity|          timestamp|             SMA_7|            SMA_30|
+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|   BTC|100937.93|100937.93|100937.93|100937.93|         1m|2025-11-06 21:37:00|         100937.93|         100937.93|
|   BTC|100922.79|100946.74|100922.79|100946.74|         1m|2025-11-06 21:38:00|100942.33499999999|100942.33499999999|
|   BTC|100930.14|100932.99|100918.83|100932.99|         1m|2025-11-06 21:39:00|100939.21999999999|100939.21999999999|
|   BTC|100898.96|100911.99|100898.96|100911.99|         1m|2025-11-06 21:40:00|100932.41249999999|100932.41249999999|
|   BTC|100924.55|100951.99|100924.54|100951.99|         1m|2025-11-06 21:41:00|        100936.328|        100936.328|
+------+---------+---------+---------+---------+

+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|Symbol|     open|     high|      low|    close|granularity|          timestamp|             SMA_7|            SMA_30|
+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|   BTC|100937.93|100946.74|100918.83|100932.99|        10m|2025-11-06 21:30:00|         100932.99|         100932.99|
|   BTC|100938.81|101146.47|100846.49|100942.81|        10m|2025-11-06 21:40:00|          100937.9|          100937.9|
|   BTC|101176.39|101212.07|100785.12|101176.39|        10m|2025-11-06 21:50:00|101017.39666666667|101017.39666666667|
|   BTC|101227.15|101394.01|101071.58|101199.42|        10m|2025-11-06 22:00:00|       101062.9025|       101062.9025|
|   BTC| 101236.0| 101236.0|101012.56|101159.51|        10m|2025-11-06 22:10:00|        101082.224|        101082.224|
+------+---------+---------+---------+---------+

[Stage 87:======================================================> (32 + 1) / 33]

+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|Symbol|     open|     high|      low|    close|granularity|          timestamp|             SMA_7|            SMA_30|
+------+---------+---------+---------+---------+-----------+-------------------+------------------+------------------+
|   BTC|101306.03|101537.99|100785.12|101346.03|         1d|2025-11-06 00:00:00|         101346.03|         101346.03|
|   BTC|100110.56|104072.05| 99315.82|103759.35|         1d|2025-11-07 00:00:00|         102552.69|         102552.69|
|   BTC|102876.01|103406.22|101472.38|103119.99|         1d|2025-11-08 00:00:00|         102741.79|         102741.79|
|   BTC| 101586.0|105447.22|101479.61|103834.58|         1d|2025-11-09 00:00:00|       103014.9875|       103014.9875|
|   BTC|106359.05|106645.43|104279.76|106286.11|         1d|2025-11-10 00:00:00|        103669.212|        103669.212|
|   BTC|105679.39| 107480.0| 102480.0|104609.01|

In [64]:
max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]
min_date = crypto_df.agg(F.min("PartitionDate")).collect()[0][0]
print(max_date)
print(min_date)

[Stage 96:====================================================>   (31 + 1) / 33]

2026-01-16
2025-11-06


In [19]:
def row_to_hbase(row):
    row_key = f"{row['Symbol']}#{row['timestamp']}#{row['granularity']}"
    return (
        row_key.encode(),
        {
            b"ohlc:open": str(row['open']).encode(),
            b"ohlc:high": str(row['high']).encode(),
            b"ohlc:low": str(row['low']).encode(),
            b"ohlc:close": str(row['close']).encode(),
            b'indicators:SMA_7': str(row['SMA_7']).encode(),
            b'indicators:SMA_30': str(row['SMA_30']).encode()
        }
    )
     
# To HBase
run_ts = datetime.now()

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')

rows = crypto_agg.collect()
max_date = crypto_df.agg(F.max("PartitionDate")).collect()[0][0]

batch_succeded = None

try:
    for row in tqdm(rows):
        key, data = row_to_hbase(row)
        table.put(key, data)
        last_successful_date = row['timestamp']
    batch_succeded = True

except Exception as e:
    print(f"Unexpected error: {e}. HBase insertion incomplete.")

finally:
    if batch_succeded is None:
        if last_successful_date:
            checkpoint_date = last_successful_date # Failed during loop
    elif batch_succeded == True:
        checkpoint_date = max_date

    new_checkpoint = spark.createDataFrame([Row(
        table_name="cryptocurrencysnapshot",
        last_processed_date=checkpoint_date,
        run_ts=run_ts
    )])

    (new_checkpoint.write
        .mode("append")
        .format("hive")
        .saveAsTable("cryptopredictions.batch_checkpoint"))

print("Checkpoint updated.")

100%|██████████| 28116/28116 [00:20<00:00, 1384.74it/s]                         


2026-01-16
2026-01-12 00:00:00
Checkpoint updated.


In [29]:
spark.stop()